# 03 - Modeling and Trade-off

This notebook explores the main utility/linkability modeling results. It is structured as an evidence trail rather than just a final table: segmentation choice, baseline models, feature selection, top-k curves, progressive feature removal and record-level aggregation.

Important framing: some analyses are final/repeated, while others are exploratory single-split analyses. The notebook keeps those roles visible.


In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.append(str(PROJECT_ROOT / "src"))

from config import FEATURE_SETS_DIR, OUTPUTS_FIGURES_DIR, OUTPUTS_TABLES_DIR

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 120)
plt.rcParams["figure.figsize"] = (8, 4)


## 1. Segmentation Selection Evidence

The main segmentation setting is `w2_o0p5` (2 s window, 1 s step). The evidence below separates the four-configuration pilot from the larger focused comparison. This supports the choice as a pragmatic operating point, not as a mathematically proven optimum.


In [ ]:
windowing_evidence_df = pd.read_csv(OUTPUTS_TABLES_DIR / "study2_windowing_selection_evidence.csv")

pilot_key = windowing_evidence_df[
    (windowing_evidence_df["evidence_stage"] == "pilot_1000_four_configs")
    & (
        ((windowing_evidence_df["task"] == "utility") & (windowing_evidence_df["model"] == "LogisticRegression"))
        | ((windowing_evidence_df["task"] == "linkability") & (windowing_evidence_df["model"] == "XGBoost"))
    )
].copy()

pilot_key["metric_label"] = pilot_key["task"] + " / " + pilot_key["model"]
pilot_key_table = pilot_key.pivot_table(
    index=["config", "window_sec", "overlap", "step_sec", "record_count", "segment_count"],
    columns="metric_label",
    values=["f1_score", "balanced_accuracy", "roc_auc"],
).reset_index()

pilot_key_table


In [ ]:
pilot_plot = pilot_key.pivot_table(index="config", columns="task", values="roc_auc")
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(pilot_plot["linkability"], pilot_plot["utility"], s=100, color="#2a9d8f")
for config, row in pilot_plot.iterrows():
    ax.annotate(config, (row["linkability"], row["utility"]), xytext=(6, 4), textcoords="offset points")
ax.set_xlabel("Linkability ROC-AUC (XGBoost)")
ax.set_ylabel("Utility ROC-AUC (Logistic Regression)")
ax.set_title("Pilot windowing trade-off, 1000 records")
plt.tight_layout()
plt.show()


In [ ]:
segmentation_selection_df = pd.read_csv(OUTPUTS_TABLES_DIR / "segmentation_selection_summary.csv")
focused_evidence_df = windowing_evidence_df[windowing_evidence_df["evidence_stage"] == "focused_5000_two_configs"].copy()

display(segmentation_selection_df)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(segmentation_selection_df["config"], segmentation_selection_df["utility_logreg_f1"], marker="o", label="Utility F1, LogReg")
ax.plot(segmentation_selection_df["config"], segmentation_selection_df["linkability_xgb_roc_auc"], marker="s", label="Linkability ROC-AUC, XGB")
ax.set_title("Focused 5000-record segmentation comparison")
ax.set_ylabel("Metric value")
ax.legend()
plt.tight_layout()
plt.show()


## 2. Final Baseline Models

The baseline comparison keeps utility and linkability separate. Utility is clinical AF/non-AF segment classification; linkability is sampled pairwise same-identifier separability.


In [ ]:
baseline_results_df = pd.read_csv(OUTPUTS_TABLES_DIR / "final_baseline_summary.csv")
baseline_results_df


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharey=False)
for ax, task in zip(axes, ["utility", "linkability"]):
    task_df = baseline_results_df[baseline_results_df["task"] == task]
    ax.bar(task_df["model"], task_df["roc_auc"], color=["#457b9d", "#e76f51"][: len(task_df)])
    ax.set_ylim(max(0.0, task_df["roc_auc"].min() - 0.05), 1.0)
    ax.set_title(f"{task.capitalize()} ROC-AUC")
    ax.tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()


## 3. Utility Feature Selection

This section inspects how utility changes as the feature subset grows. The `top-150` subset should be interpreted as exploratory/single-split unless recomputed with nested or multi-seed selection.


In [ ]:
utility_feature_selection_summary = pd.read_csv(FEATURE_SETS_DIR / "utility_feature_selection_summary.csv")
utility_feature_selection_summary[["ranking_model", "model", "subset_size", "f1_score", "balanced_accuracy", "roc_auc", "pr_auc"]]


In [ ]:
logreg_subset_curve = utility_feature_selection_summary[
    utility_feature_selection_summary["model"] == "LogisticRegression"
].sort_values("subset_size")

fig, ax = plt.subplots(figsize=(8, 4))
for metric, color in [("f1_score", "#457b9d"), ("balanced_accuracy", "#2a9d8f"), ("roc_auc", "#e76f51")]:
    ax.plot(
        logreg_subset_curve["subset_size"],
        logreg_subset_curve[metric],
        marker="o",
        label=metric,
        color=color,
    )
ax.set_xlabel("Number of selected features")
ax.set_ylabel("Utility metric")
ax.set_title("Utility feature subset curve - Logistic Regression baseline")
ax.legend()
plt.tight_layout()
plt.show()

model_comparison_curve = utility_feature_selection_summary.sort_values(["model", "subset_size"])
fig, axes = plt.subplots(1, 2, figsize=(11, 4), sharex=True)
for model_name, group in model_comparison_curve.groupby("model"):
    axes[0].plot(group["subset_size"], group["f1_score"], marker="o", label=model_name)
    axes[1].plot(group["subset_size"], group["roc_auc"], marker="o", label=model_name)
axes[0].set_title("F1 by model")
axes[0].set_ylabel("F1")
axes[1].set_title("ROC-AUC by model")
axes[1].set_ylabel("ROC-AUC")
for ax in axes:
    ax.set_xlabel("Number of selected features")
    ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
top150_features = pd.read_csv(FEATURE_SETS_DIR / "utility_top150_features.csv")
feature_col = "feature" if "feature" in top150_features.columns else top150_features.columns[0]

def feature_family(feature_name: str) -> str:
    if feature_name.startswith("lead_"):
        parts = feature_name.split("_")
        return "_".join(parts[:2])
    if feature_name.startswith("global_"):
        return "global"
    return "other"

feature_family_df = (
    top150_features[feature_col]
    .map(feature_family)
    .value_counts()
    .rename_axis("feature_family")
    .reset_index(name="n_top150_features")
)

display(top150_features.head(20))
display(feature_family_df)

feature_family_df.plot.bar(x="feature_family", y="n_top150_features", legend=False, color="#577590", figsize=(10, 4))
plt.title("Feature families represented in the top-150 utility subset")
plt.xlabel("Feature family")
plt.ylabel("Number of features")
plt.tight_layout()
plt.show()


## 4. Top-k Trade-off Curve

This curve asks a direct question: as we add more utility-ranked features, how quickly does utility recover and how quickly does linkability also increase?


In [ ]:
topk_tradeoff_df = pd.read_csv(OUTPUTS_TABLES_DIR / "topk_utility_linkability_curve.csv")

display(topk_tradeoff_df)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(topk_tradeoff_df["top_k"], topk_tradeoff_df["utility_f1"], marker="o", color="#457b9d", label="Utility F1")
ax2.plot(topk_tradeoff_df["top_k"], topk_tradeoff_df["linkability_roc_auc"], marker="s", color="#d62828", label="Linkability ROC-AUC")
ax1.set_xlabel("Top-k utility-ranked features")
ax1.set_ylabel("Utility F1", color="#457b9d")
ax2.set_ylabel("Linkability ROC-AUC", color="#d62828")
ax1.set_title("Top-k features vs utility/linkability")
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="lower right")
plt.tight_layout()
plt.show()


## 5. Feature Importance Overlap by Task

This section checks whether the same feature families support AF/non-AF utility and segment linkability. The current artefacts use the full final segment-feature dataset and provide a task-level map of which feature families support utility versus linkability. The rankings are still model-based and single-seed, so they should be treated as directional evidence rather than definitive causal attribution.

Recompute the artefacts with:

```powershell
.\venv\Scripts\python.exe src\run_feature_group_importance_analysis.py --max-chunks 0 --run-name feature_group_importance_full
```


In [ ]:
feature_importance_run = "feature_group_importance_full"
feature_importance_paths = {
    "importance": OUTPUTS_TABLES_DIR / f"{feature_importance_run}_feature_importance_long.csv",
    "groups": OUTPUTS_TABLES_DIR / f"{feature_importance_run}_group_summary.csv",
    "overlap": OUTPUTS_TABLES_DIR / f"{feature_importance_run}_feature_overlap.csv",
    "candidates": OUTPUTS_TABLES_DIR / f"{feature_importance_run}_customization_candidates.csv",
    "protocol": OUTPUTS_TABLES_DIR / f"{feature_importance_run}_protocol.json",
}

missing_paths = [str(path) for path in feature_importance_paths.values() if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Missing feature-importance artefacts. Run src/run_feature_group_importance_analysis.py first. "
        f"Missing: {missing_paths}"
    )

with open(feature_importance_paths["protocol"], encoding="utf-8") as fp:
    feature_importance_protocol = json.load(fp)

feature_importance_long_df = pd.read_csv(feature_importance_paths["importance"])
feature_group_summary_df = pd.read_csv(feature_importance_paths["groups"])
feature_overlap_df = pd.read_csv(feature_importance_paths["overlap"])
customization_candidates_df = pd.read_csv(feature_importance_paths["candidates"])

protocol_summary = pd.DataFrame([
    {
        "loaded_chunks": feature_importance_protocol["loaded_chunks"],
        "loaded_segments": feature_importance_protocol["loaded_segments"],
        "loaded_identifiers": feature_importance_protocol["loaded_identifiers"],
        "n_features": feature_importance_protocol["n_features"],
        "utility_model": feature_importance_protocol["utility_model"],
        "linkability_model": feature_importance_protocol["linkability_model"],
        "linkability_pairs": feature_importance_protocol["linkability_protocol"]["max_pairs"],
        "min_segment_gap": feature_importance_protocol["linkability_protocol"]["min_segment_gap"],
    }
])
display(protocol_summary)


### Group-Level Signal

These plots aggregate model-based importance by interpretable groups. A group that is high for both tasks is a privacy-utility conflict zone; a group much higher for linkability is a stronger candidate for heavier protection.


In [ ]:
OUTPUTS_FIGURES_DIR.mkdir(parents=True, exist_ok=True)

stat_group_pivot = (
    feature_group_summary_df[feature_group_summary_df["group_type"] == "stat_group"]
    .pivot_table(index="group_value", columns="task", values="importance_share", aggfunc="sum")
    .fillna(0.0)
    .sort_values("linkability", ascending=False)
)
display(stat_group_pivot.round(3))

fig, ax = plt.subplots(figsize=(8, 4))
stat_group_pivot.plot.bar(ax=ax, color={"utility": "#3A6EA5", "linkability": "#C44E52"})
ax.set_title("Feature-group importance by task")
ax.set_xlabel("Feature group")
ax.set_ylabel("Share of model-based importance")
ax.legend(title="Task")
fig.tight_layout()
fig.savefig(OUTPUTS_FIGURES_DIR / "feature_importance_stat_groups.png", dpi=200, bbox_inches="tight")
plt.show()

lead_group_pivot = (
    feature_group_summary_df[feature_group_summary_df["group_type"] == "lead_or_global"]
    .pivot_table(index="group_value", columns="task", values="importance_share", aggfunc="sum")
    .fillna(0.0)
)
lead_group_pivot["shared_min"] = lead_group_pivot[["utility", "linkability"]].min(axis=1)
lead_group_pivot = lead_group_pivot.sort_values("shared_min", ascending=False).drop(columns="shared_min")
display(lead_group_pivot.head(12).round(3))

fig, ax = plt.subplots(figsize=(9, 4))
lead_group_pivot.head(12).plot.bar(ax=ax, color={"utility": "#3A6EA5", "linkability": "#C44E52"})
ax.set_title("Top shared leads/global aggregates")
ax.set_xlabel("Lead or global aggregate")
ax.set_ylabel("Share of model-based importance")
ax.legend(title="Task")
fig.tight_layout()
fig.savefig(OUTPUTS_FIGURES_DIR / "feature_importance_lead_groups.png", dpi=200, bbox_inches="tight")
plt.show()


### Feature-Level Overlap

The diagonal separates features that contribute similarly to both tasks from features that are more task-specific. Points above the diagonal are more linkability-dominant; points below are more utility-dominant.


In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
scatter = ax.scatter(
    feature_overlap_df["utility"],
    feature_overlap_df["linkability"],
    c=feature_overlap_df["shared_min_importance"],
    cmap="viridis",
    alpha=0.8,
    s=42,
)
limit = max(feature_overlap_df["utility"].max(), feature_overlap_df["linkability"].max()) * 1.08
ax.plot([0, limit], [0, limit], linestyle="--", color="#333333", linewidth=1)
for _, row in feature_overlap_df.sort_values("shared_min_importance", ascending=False).head(10).iterrows():
    ax.annotate(row["feature"], (row["utility"], row["linkability"]), fontsize=8, alpha=0.85)
ax.set_xlim(0, limit)
ax.set_ylim(0, limit)
ax.set_xlabel("Utility importance share")
ax.set_ylabel("Linkability importance share")
ax.set_title("Feature-level utility vs linkability importance")
fig.colorbar(scatter, ax=ax, label="Shared minimum importance")
fig.tight_layout()
fig.savefig(OUTPUTS_FIGURES_DIR / "feature_importance_overlap_scatter.png", dpi=200, bbox_inches="tight")
plt.show()

display(
    feature_overlap_df.sort_values("shared_min_importance", ascending=False)
    .head(20)[[
        "feature",
        "utility",
        "linkability",
        "shared_min_importance",
        "linkability_minus_utility",
        "feature_scope",
        "lead_or_global",
        "feature_stat",
        "stat_group",
    ]]
)


### Candidate Sets for Selective Privacy

`shared_high` features require care because transforming them may reduce utility and linkability together. `linkability_dominant` features are the first candidates for stronger privacy transformations. `utility_dominant` features are candidates to preserve or transform more lightly.


In [ ]:
candidate_columns = [
    "candidate_type",
    "feature",
    "utility",
    "linkability",
    "shared_min_importance",
    "linkability_minus_utility",
    "utility_minus_linkability",
    "lead_or_global",
    "feature_stat",
    "stat_group",
]
display(customization_candidates_df.groupby("candidate_type", group_keys=False).head(10)[candidate_columns])

candidate_group_summary = (
    customization_candidates_df
    .groupby(["candidate_type", "stat_group"], as_index=False)
    .agg(n_features=("feature", "nunique"), mean_utility=("utility", "mean"), mean_linkability=("linkability", "mean"))
    .sort_values(["candidate_type", "n_features", "mean_linkability"], ascending=[True, False, False])
)
display(candidate_group_summary)


## 6. Same-Data Comparison: Full Features vs Top-150

This is the compact version of the feature-subset result. It compares the full 208-feature representation with the selected top-150 subset on the same split.


In [ ]:
same_data_tradeoff_df = pd.read_csv(OUTPUTS_TABLES_DIR / "same_data_tradeoff_summary.csv")
same_data_tradeoff_df.assign(
    delta_utility_f1=lambda df: df["utility_f1"] - df.loc[df["feature_set"] == "full_208", "utility_f1"].iloc[0],
    delta_linkability_xgb_roc_auc=lambda df: df["linkability_xgb_roc_auc"] - df.loc[df["feature_set"] == "full_208", "linkability_xgb_roc_auc"].iloc[0],
)


## 7. Progressive Feature Removal

Starting from the utility-oriented top-150 subset, features with high linkability importance are progressively removed. This is useful for discussing trade-off sensitivity, but remains exploratory unless repeated with nested/multi-seed selection.


In [ ]:
feature_removal_tradeoff_df = pd.read_csv(OUTPUTS_TABLES_DIR / "feature_removal_tradeoff_summary.csv")

display(feature_removal_tradeoff_df)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.plot(feature_removal_tradeoff_df["n_removed"], feature_removal_tradeoff_df["utility_f1"], marker="o", color="#457b9d", label="Utility F1")
ax2.plot(feature_removal_tradeoff_df["n_removed"], feature_removal_tradeoff_df["linkability_roc_auc"], marker="s", color="#d62828", label="Linkability ROC-AUC")
ax1.set_xlabel("Features removed from top-150")
ax1.set_ylabel("Utility F1", color="#457b9d")
ax2.set_ylabel("Linkability ROC-AUC", color="#d62828")
ax1.set_title("Progressive feature removal trade-off")
lines_1, labels_1 = ax1.get_legend_handles_labels()
lines_2, labels_2 = ax2.get_legend_handles_labels()
ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc="best")
plt.tight_layout()
plt.show()


## 8. Segment-Level vs Record-Level Utility

The default utility task classifies individual segments. This check asks whether aggregating segment evidence at record level makes the clinical task easier.


In [ ]:
record_level_utility_df = pd.read_csv(OUTPUTS_TABLES_DIR / "record_level_utility_summary.csv")

display(record_level_utility_df)

record_level_utility_df.plot.bar(x="representation", y=["f1_score", "balanced_accuracy", "roc_auc"], figsize=(9, 4))
plt.title("Segment-level vs record-level utility")
plt.xlabel("Representation")
plt.ylabel("Metric")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()


## Main Takeaways

- `w2_o0p5` is retained as a pragmatic operating point: `w3_o0p5` improves utility but also increases linkability.
- Logistic Regression is the selected utility baseline; XGBoost is the selected linkability baseline.
- Utility recovers strongly as feature count increases, but linkability rises quickly too.
- Feature-importance overlap identifies privacy-utility conflict zones: shared high-importance features need cautious transformation, while linkability-dominant features are better candidates for stronger protection.
- The top-150, feature-overlap, and progressive-removal experiments are valuable for trade-off exploration, but should be described as exploratory unless recomputed with nested/multi-seed selection.
- Record-level aggregation improves utility, suggesting the unit of analysis matters substantially.
